# Vanguard Digital Experiment — Phase 3: Hypothesis Testing

## Objective

Phase 2 showed that the redesigned digital experience (**Test**) performed differently from the traditional experience (**Control**) across several KPIs. This notebook validates whether those observed differences are statistically reliable and meaningful for Vanguard's business decision.

We focus on four analytical questions:

1. **Completion rate:** Did the redesigned UI significantly increase completion?
2. **Business threshold:** Did the improvement exceed Vanguard's 5 percentage-point cost-effectiveness threshold?
3. **Experiment validity:** Were Test and Control comparable before treatment?
4. **User friction and efficiency:** Did the redesign affect error rate and time-to-complete?

**Significance level:** `alpha = 0.05`  
**Main unit of analysis:** one row per `visit_id`  
**Demographic balance checks:** one row per `client_id`

> This notebook uses the cleaned CSVs generated in Phase 1. It does not reload or re-clean raw files, keeping the project workflow modular and reproducible.

In [1]:
# Core libraries
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Notebook settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

import warnings
warnings.filterwarnings("ignore")

ALPHA = 0.05
THRESHOLD = 0.05
STEP_ORDER = {"start": 0, "step_1": 1, "step_2": 2, "step_3": 3, "confirm": 4}
GROUP_COLORS = {"Control": "#4C78A8", "Test": "#F58518"}

## 1. Load Cleaned Analytical Dataset

The cleaned merged dataset contains web events, experiment assignment, client demographics, and engineered segmentation variables. It is the master dataset created during the data cleaning phase.

In [2]:
# Load cleaned analytical dataset

df = pd.read_csv("vanguard_cleaned_merged.csv")

df['date_time'] = pd.to_datetime(df['date_time'])

print(df.shape)
df.head()

(317235, 19)


,client_id,visitor_id,visit_id,process_step,date_time,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,age_group,tenure_group,balance_segment,date,year_month
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0000,64.0000,79.0000,Unknown,2.0000,"189,023.8600",1.0000,4.0000,70+,New,Very High,2017-04-17,2017-04
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0000,64.0000,79.0000,Unknown,2.0000,"189,023.8600",1.0000,4.0000,70+,New,Very High,2017-04-17,2017-04
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0000,64.0000,79.0000,Unknown,2.0000,"189,023.8600",1.0000,4.0000,70+,New,Very High,2017-04-17,2017-04
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0000,64.0000,79.0000,Unknown,2.0000,"189,023.8600",1.0000,4.0000,70+,New,Very High,2017-04-17,2017-04
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0000,64.0000,79.0000,Unknown,2.0000,"189,023.8600",1.0000,4.0000,70+,New,Very High,2017-04-17,2017-04


### Data Readiness Check

Before running statistical tests, we confirm that the cleaned dataset contains the required fields and that the experiment groups are available.

In [3]:
required_columns = [
    "client_id", "visit_id", "process_step", "date_time", "Variation",
    "clnt_age", "clnt_tenure_yr", "bal"
]

missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("Required columns are present.")
print("Experiment group counts:")
display(df["Variation"].value_counts(dropna=False))

print("Process step counts:")
display(df["process_step"].value_counts())

Required columns are present.
Experiment group counts:


Variation
Test       176699
Control    140536
Name: count, dtype: int64

Process step counts:


process_step
start      101153
step_1      68210
step_2      56672
step_3      48264
confirm     42936
Name: count, dtype: int64

## 2. Build Visit-Level Dataset

The experiment evaluates whether users complete the online process. Since users may generate multiple web events inside one visit, we summarize the event-level data into one row per `visit_id`.

This visit-level table is used for completion, error-rate, and time-to-complete testing.

In [4]:
# Keep only valid experimental observations and known process steps
analysis_df = df.dropna(subset=["Variation", "visit_id", "process_step", "date_time"]).copy()
analysis_df["step_index"] = analysis_df["process_step"].map(STEP_ORDER)
analysis_df = analysis_df.dropna(subset=["step_index"]).copy()
analysis_df["step_index"] = analysis_df["step_index"].astype(int)

# Sort events chronologically within each visit
analysis_df = analysis_df.sort_values(["client_id", "visit_id", "date_time"]).reset_index(drop=True)

# One row per visit
visit = analysis_df.groupby("visit_id").agg(
    Variation=("Variation", "first"),
    client_id=("client_id", "first"),
    reached_confirm=("process_step", lambda steps: "confirm" in set(steps)),
    max_step=("step_index", "max"),
    start_time=("date_time", "min"),
    end_time=("date_time", "max"),
    total_events=("process_step", "size")
).reset_index()

print(f"Event-level rows: {analysis_df.shape[0]:,}")
print(f"Visit-level rows: {visit.shape[0]:,}")
visit.head()

Event-level rows: 317,235
Visit-level rows: 69,205


,visit_id,Variation,client_id,reached_confirm,max_step,start_time,end_time,total_events
0,100012776_37918976071_457913,Test,3561384,True,4,2017-04-26 13:22:17,2017-04-26 13:23:09,2
1,100019538_17884295066_43909,Test,7338123,True,4,2017-04-09 16:20:56,2017-04-09 16:24:58,11
2,100022086_87870757897_149620,Test,2478628,True,4,2017-05-23 20:44:01,2017-05-23 20:47:01,5
3,100030127_47967100085_936361,Control,105007,False,0,2017-03-22 11:07:49,2017-03-22 11:07:49,1
4,100037962_47432393712_705583,Control,5623007,False,1,2017-04-14 16:41:51,2017-04-14 16:44:03,4


## 3. Completion Rate Inputs

Completion rate is calculated as the proportion of visits that reached the `confirm` step.

In [5]:
comp = visit.groupby("Variation")["reached_confirm"].agg(n="size", completed="sum")
comp["completion_rate"] = comp["completed"] / comp["n"]
comp["completion_rate_pct"] = comp["completion_rate"] * 100
comp

,n,completed,completion_rate,completion_rate_pct
Variation,,,,
Control,32128,15999,0.4980,49.7977
Test,37077,21681,0.5848,58.4756


## 4. Statistical Helper Functions

We use two-proportion z-tests for completion and error-rate comparisons. For the 5 percentage-point threshold test, we compare the observed difference against a non-zero null difference.

In [6]:
def two_prop_ztest(x1, n1, x2, n2, alternative="larger", null_diff=0.0, pooled=True):
    """
    Run a two-proportion z-test for (p1 - p2) against a null difference.

    Parameters
    ----------
    x1, x2 : int
        Number of successes in group 1 and group 2.
    n1, n2 : int
        Number of observations in group 1 and group 2.
    alternative : str
        'larger', 'smaller', or 'two-sided'.
    null_diff : float
        Difference under the null hypothesis.
    pooled : bool
        Whether to use pooled standard error. Usually True for null_diff = 0.

    Returns
    -------
    dict
        p1, p2, difference, standard error, z-statistic and p-value.
    """
    p1, p2 = x1 / n1, x2 / n2
    diff = p1 - p2

    if pooled and null_diff == 0:
        pooled_p = (x1 + x2) / (n1 + n2)
        se = np.sqrt(pooled_p * (1 - pooled_p) * (1 / n1 + 1 / n2))
    else:
        se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)

    z = (diff - null_diff) / se

    if alternative == "larger":
        p_value = 1 - stats.norm.cdf(z)
    elif alternative == "smaller":
        p_value = stats.norm.cdf(z)
    elif alternative == "two-sided":
        p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    else:
        raise ValueError("alternative must be 'larger', 'smaller', or 'two-sided'")

    return {"p1": p1, "p2": p2, "diff": diff, "se": se, "z": z, "p_value": p_value}


def diff_ci(x1, n1, x2, n2, confidence=0.95):
    """Return a confidence interval for the difference between two proportions."""
    p1, p2 = x1 / n1, x2 / n2
    diff = p1 - p2
    se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    z_crit = stats.norm.ppf(1 - (1 - confidence) / 2)
    return diff - z_crit * se, diff + z_crit * se


def format_pvalue(p):
    """Format very small p-values for readability."""
    return "< 0.001" if p < 0.001 else f"{p:.4f}"


def decision_text(p, alpha=ALPHA):
    """Return hypothesis test decision based on p-value and alpha."""
    return "Reject H0" if p < alpha else "Fail to reject H0"


def cohen_h(p1, p2):
    """Cohen's h effect size for two proportions."""
    return 2 * np.arcsin(np.sqrt(p1)) - 2 * np.arcsin(np.sqrt(p2))

## 5. Hypothesis Test 1 — Completion Rate

**Business question:** Did the redesigned interface lead to a higher completion rate?

- **H0:** completion_rate(Test) = completion_rate(Control)
- **H1:** completion_rate(Test) > completion_rate(Control)
- **Test:** two-proportion z-test, one-sided
- **Alpha:** 0.05

In [7]:
x_test, n_test = comp.loc["Test", "completed"], comp.loc["Test", "n"]
x_control, n_control = comp.loc["Control", "completed"], comp.loc["Control", "n"]

completion_result = two_prop_ztest(
    x_test, n_test, x_control, n_control,
    alternative="larger",
    pooled=True
)

completion_ci_low, completion_ci_high = diff_ci(x_test, n_test, x_control, n_control)
completion_effect = cohen_h(completion_result["p1"], completion_result["p2"])

print("TEST 1 — Completion Rate: Test > Control")
print("-" * 55)
print(f"Test completion rate    : {completion_result['p1']:.4f} ({completion_result['p1']:.1%})")
print(f"Control completion rate : {completion_result['p2']:.4f} ({completion_result['p2']:.1%})")
print(f"Difference              : {completion_result['diff']*100:+.2f} percentage points")
print(f"95% CI for difference   : [{completion_ci_low*100:.2f}, {completion_ci_high*100:.2f}] pp")
print(f"Cohen's h               : {completion_effect:.4f}")
print(f"z-statistic             : {completion_result['z']:.3f}")
print(f"p-value                 : {format_pvalue(completion_result['p_value'])}")
print("-" * 55)
print(decision_text(completion_result["p_value"]))

TEST 1 — Completion Rate: Test > Control
-------------------------------------------------------
Test completion rate    : 0.5848 (58.5%)
Control completion rate : 0.4980 (49.8%)
Difference              : +8.68 percentage points
95% CI for difference   : [7.94, 9.42] pp
Cohen's h               : 0.1744
z-statistic             : 22.861
p-value                 : < 0.001
-------------------------------------------------------
Reject H0


### Interpretation — Test 1

If the p-value is below 0.05, we reject the null hypothesis and conclude that the redesigned interface produced a statistically significant increase in completion rate. The confidence interval helps quantify the likely range of the improvement in percentage points.

## 6. Hypothesis Test 2 — 5 Percentage-Point Business Threshold

A statistically significant improvement is not automatically enough to justify implementation. Vanguard defined a cost-effectiveness threshold: the redesign should improve completion by **at least 5 percentage points**.

- **H0:** completion_rate(Test) − completion_rate(Control) = 5 percentage points
- **H1:** completion_rate(Test) − completion_rate(Control) > 5 percentage points
- **Test:** two-proportion z-test, one-sided, unpooled standard error
- **Alpha:** 0.05

In [9]:
threshold_result = two_prop_ztest(
    x_test, n_test, x_control, n_control,
    alternative="larger",
    null_diff=THRESHOLD,
    pooled=False
)

relative_threshold = completion_result["p2"] * 1.05

print("TEST 2 — Completion Lift Exceeds 5 Percentage Points")
print("-" * 62)
print(f"Observed lift           : {threshold_result['diff']*100:+.2f} percentage points")
print(f"Business threshold      : {THRESHOLD*100:.1f} percentage points")
print(f"Excess over threshold   : {(threshold_result['diff'] - THRESHOLD)*100:+.2f} percentage points")
print(f"z-statistic             : {threshold_result['z']:.3f}")
print(f"p-value                 : {format_pvalue(threshold_result['p_value'])}")
print("-" * 62)
print(decision_text(threshold_result["p_value"]))

print("Relative 5% sanity check:")
print(f"Control x 1.05 = {relative_threshold:.4f}")
print(f"Test rate      = {completion_result['p1']:.4f}")
print("Test is above the relative 5% threshold." if completion_result["p1"] > relative_threshold else "Test is below the relative 5% threshold.")

TEST 2 — Completion Lift Exceeds 5 Percentage Points
--------------------------------------------------------------
Observed lift           : +8.68 percentage points
Business threshold      : 5.0 percentage points
Excess over threshold   : +3.68 percentage points
z-statistic             : 9.716
p-value                 : < 0.001
--------------------------------------------------------------
Reject H0
Relative 5% sanity check:
Control x 1.05 = 0.5229
Test rate      = 0.5848
Test is above the relative 5% threshold.


### Interpretation — Test 2

This test connects statistical evidence to business action. If the p-value is below 0.05, the observed uplift is not only statistically significant, but also strong enough to clear Vanguard's 5 percentage-point threshold.

## 7. Additional Hypothesis Test A — Randomization / Age Balance

A valid A/B test depends on comparable groups before treatment. If Test and Control differ substantially in client characteristics, the observed performance difference may reflect group composition rather than the interface redesign.

Here we test whether mean client age differs between Test and Control.

- **H0:** mean_age(Test) = mean_age(Control)
- **H1:** mean_age(Test) ≠ mean_age(Control)
- **Test:** Welch's independent t-test
- **Unit of analysis:** one row per client

In [10]:
client_demo = (
    analysis_df
    .drop_duplicates("client_id")
    [["client_id", "Variation", "clnt_age", "clnt_tenure_yr", "bal"]]
    .dropna(subset=["Variation", "clnt_age"])
)

age_test = client_demo.loc[client_demo["Variation"] == "Test", "clnt_age"]
age_control = client_demo.loc[client_demo["Variation"] == "Control", "clnt_age"]

age_t_stat, age_p_value = stats.ttest_ind(age_test, age_control, equal_var=False)
age_diff = age_test.mean() - age_control.mean()

print("TEST 3A — Age Balance: Test vs Control")
print("-" * 50)
print(f"Mean age Test    : {age_test.mean():.2f} years (n={len(age_test):,})")
print(f"Mean age Control : {age_control.mean():.2f} years (n={len(age_control):,})")
print(f"Difference       : {age_diff:+.2f} years")
print(f"t-statistic      : {age_t_stat:.3f}")
print(f"p-value          : {format_pvalue(age_p_value)}")
print("-" * 50)
print(decision_text(age_p_value))

TEST 3A — Age Balance: Test vs Control
--------------------------------------------------
Mean age Test    : 47.16 years (n=26,961)
Mean age Control : 47.50 years (n=23,526)
Difference       : -0.33 years
t-statistic      : -2.416
p-value          : 0.0157
--------------------------------------------------
Reject H0


### Interpretation — Age Balance

With large samples, very small differences can become statistically significant. For experiment evaluation, the practical size of the difference matters. A difference of only a fraction of a year would not represent a meaningful randomization problem, even if the p-value is below 0.05.

## 8. Additional Hypothesis Test B — Error Rate

In this project, an error is defined as a backward move in the process, such as going from `step_2` back to `step_1`. This behavior is a proxy for friction or confusion.

- **H0:** error_rate(Test) = error_rate(Control)
- **H1:** error_rate(Test) ≠ error_rate(Control)
- **Test:** two-proportion z-test, two-sided
- **Unit of analysis:** one row per visit

In [11]:
analysis_df["previous_step_index"] = analysis_df.groupby("visit_id")["step_index"].shift(1)
analysis_df["is_backward"] = analysis_df["step_index"] < analysis_df["previous_step_index"]

visit_errors = analysis_df.groupby("visit_id").agg(
    Variation=("Variation", "first"),
    backward_moves=("is_backward", "sum")
).reset_index()

visit_errors["has_error"] = visit_errors["backward_moves"] > 0

error_summary = visit_errors.groupby("Variation")["has_error"].agg(n="size", errors="sum")
error_summary["error_rate"] = error_summary["errors"] / error_summary["n"]
error_summary

,n,errors,error_rate
Variation,,,
Control,32128,6584,0.2049
Test,37077,10037,0.2707


In [12]:
errors_test, error_n_test = error_summary.loc["Test", "errors"], error_summary.loc["Test", "n"]
errors_control, error_n_control = error_summary.loc["Control", "errors"], error_summary.loc["Control", "n"]

error_result = two_prop_ztest(
    errors_test, error_n_test,
    errors_control, error_n_control,
    alternative="two-sided",
    pooled=True
)

error_effect = cohen_h(error_result["p1"], error_result["p2"])

print("TEST 3B — Error Rate: Test vs Control")
print("-" * 50)
print(f"Test error rate    : {error_result['p1']:.4f} ({error_result['p1']:.1%})")
print(f"Control error rate : {error_result['p2']:.4f} ({error_result['p2']:.1%})")
print(f"Difference         : {error_result['diff']*100:+.2f} percentage points")
print(f"Cohen's h          : {error_effect:.4f}")
print(f"z-statistic        : {error_result['z']:.3f}")
print(f"p-value            : {format_pvalue(error_result['p_value'])}")
print("-" * 50)
print(decision_text(error_result["p_value"]))

TEST 3B — Error Rate: Test vs Control
--------------------------------------------------
Test error rate    : 0.2707 (27.1%)
Control error rate : 0.2049 (20.5%)
Difference         : +6.58 percentage points
Cohen's h          : 0.1548
z-statistic        : 20.201
p-value            : < 0.001
--------------------------------------------------
Reject H0


### Interpretation — Error Rate

If the Test group has a significantly higher error rate, the redesign may be improving completion while still creating some navigation friction. This is a UX trade-off worth flagging for the design team.

## 9. Additional Hypothesis Test C — Time to Complete

Among visits that reached `confirm`, we compare the total time from the first event to the last event. Since completion times are typically skewed, we use a non-parametric Mann-Whitney U test.

- **H0:** time-to-complete distributions are equal between Test and Control
- **H1:** time-to-complete distributions differ between Test and Control
- **Test:** Mann-Whitney U test, two-sided
- **Unit of analysis:** completed visits only

In [13]:
completed_visits = visit[visit["reached_confirm"]].copy()
completed_visits["total_minutes"] = (
    completed_visits["end_time"] - completed_visits["start_time"]
).dt.total_seconds() / 60

# Remove impossible or missing durations if any
completed_visits = completed_visits.dropna(subset=["total_minutes"])
completed_visits = completed_visits[completed_visits["total_minutes"] >= 0]

time_test = completed_visits.loc[completed_visits["Variation"] == "Test", "total_minutes"]
time_control = completed_visits.loc[completed_visits["Variation"] == "Control", "total_minutes"]

time_u_stat, time_p_value = stats.mannwhitneyu(time_test, time_control, alternative="two-sided")

print("TEST 3C — Time to Complete: Test vs Control")
print("-" * 55)
print(f"Median minutes Test    : {time_test.median():.2f}")
print(f"Median minutes Control : {time_control.median():.2f}")
print(f"Median difference      : {time_test.median() - time_control.median():+.2f} minutes")
print(f"U-statistic            : {time_u_stat:,.0f}")
print(f"p-value                : {format_pvalue(time_p_value)}")
print("-" * 55)
print(decision_text(time_p_value))

TEST 3C — Time to Complete: Test vs Control
-------------------------------------------------------
Median minutes Test    : 3.58
Median minutes Control : 4.48
Median difference      : -0.90 minutes
U-statistic            : 147,538,108
p-value                : < 0.001
-------------------------------------------------------
Reject H0


### Interpretation — Time to Complete

A lower median time-to-complete for the Test group suggests that users who successfully complete the redesigned process do so more efficiently. This should be interpreted together with error rate, because a faster process is strongest when it does not increase user friction.

## 10. Executive Results Table

This table converts statistical test outputs into business interpretation. It is designed to be reusable in the README and final presentation.

In [14]:
summary_table = pd.DataFrame({
    "Hypothesis Test": [
        "Completion Rate",
        "5 pp Threshold",
        "Age Balance",
        "Error Rate",
        "Time to Complete"
    ],
    "Statistical Test": [
        "Two-proportion z-test (one-sided)",
        "Two-proportion z-test vs 5 pp threshold",
        "Welch's t-test (two-sided)",
        "Two-proportion z-test (two-sided)",
        "Mann-Whitney U test (two-sided)"
    ],
    "Observed Result": [
        f"Test {completion_result['p1']:.1%} vs Control {completion_result['p2']:.1%} ({completion_result['diff']*100:+.2f} pp)",
        f"Observed lift {threshold_result['diff']*100:+.2f} pp vs {THRESHOLD*100:.1f} pp threshold",
        f"Mean age gap {age_diff:+.2f} years",
        f"Test {error_result['p1']:.1%} vs Control {error_result['p2']:.1%} ({error_result['diff']*100:+.2f} pp)",
        f"Median Test {time_test.median():.2f} min vs Control {time_control.median():.2f} min"
    ],
    "P-Value": [
        format_pvalue(completion_result["p_value"]),
        format_pvalue(threshold_result["p_value"]),
        format_pvalue(age_p_value),
        format_pvalue(error_result["p_value"]),
        format_pvalue(time_p_value)
    ],
    "Decision": [
        decision_text(completion_result["p_value"]),
        decision_text(threshold_result["p_value"]),
        decision_text(age_p_value),
        decision_text(error_result["p_value"]),
        decision_text(time_p_value)
    ],
    "Business Interpretation": [
        "The redesigned UI significantly improves completion.",
        "The uplift clears Vanguard's cost-effectiveness bar.",
        "The age gap is statistically detectable but practically negligible.",
        "The Test group shows higher backward navigation; monitor UX friction.",
        "Completed Test visits are faster, indicating better process efficiency."
    ]
})

summary_table

,Hypothesis Test,Statistical Test,Observed Result,P-Value,Decision,Business Interpretation
0,Completion Rate,Two-proportion z-test (one-sided),Test 58.5% vs Control 49.8% (+8.68 pp),< 0.001,Reject H0,The redesigned UI significantly improves compl...
1,5 pp Threshold,Two-proportion z-test vs 5 pp threshold,Observed lift +8.68 pp vs 5.0 pp threshold,< 0.001,Reject H0,The uplift clears Vanguard's cost-effectivenes...
2,Age Balance,Welch's t-test (two-sided),Mean age gap -0.33 years,0.0157,Reject H0,The age gap is statistically detectable but pr...
3,Error Rate,Two-proportion z-test (two-sided),Test 27.1% vs Control 20.5% (+6.58 pp),< 0.001,Reject H0,The Test group shows higher backward navigatio...
4,Time to Complete,Mann-Whitney U test (two-sided),Median Test 3.58 min vs Control 4.48 min,< 0.001,Reject H0,"Completed Test visits are faster, indicating b..."


## 11. Final Statistical Conclusions

### Completion Rate

The redesigned interface produced a statistically significant increase in completion rate. The Test group achieved a higher completion rate than the Control group, and the estimated uplift is large enough to be meaningful for the business.

### Cost-Effectiveness Threshold

The observed lift exceeds Vanguard's 5 percentage-point threshold. This means the improvement is not only statistically significant, but also strong enough to support a business case for broader rollout.

### Experiment Validity

The age balance test may detect a statistically significant difference because of the large sample size, but the observed difference is very small in practical terms. This does not appear to be a meaningful randomization concern.

### User Friction

The redesign also shows a trade-off: the Test group has a significantly higher backward-navigation rate. This suggests that while more users complete the process, some parts of the new experience may still create confusion or require additional refinement.

### Process Efficiency

Among completed visits, the Test group finishes faster than the Control group. This supports the idea that the redesigned experience improves efficiency for users who successfully complete the journey.

## 12. Business Recommendation

Based on the statistical evidence, Vanguard should consider moving forward with broader deployment of the redesigned interface because it significantly increases completion rate and exceeds the 5 percentage-point business threshold.

However, the rollout should be accompanied by targeted UX monitoring. The higher backward-navigation rate in the Test group suggests that some steps may still generate friction, even though the overall process performs better.

**Recommended next actions:**

1. Roll out the redesigned UI gradually while monitoring error-related KPIs.
2. Investigate which specific steps generate the most backward navigation.
3. Collect additional behavioral data such as device type, browser, rage clicks, and user satisfaction.
4. Run a follow-up experiment focused specifically on reducing friction inside the redesigned journey.

**Bottom line:** the redesign wins on completion and efficiency, but Vanguard should refine the experience to reduce navigation friction before a full-scale permanent rollout.